# SDXL-Turbo: alternative dataset bounds versus Parti P2

Run the six dataset notebooks first. This notebook compares observed bounds without further model inference. The original notebooks and packaged clamp tensors are preserved.

In [ ]:
import sys
from pathlib import Path

EXPERIMENT_DIR = Path.cwd()
if not (EXPERIMENT_DIR / "dataset_bounds.py").exists():
    EXPERIMENT_DIR = EXPERIMENT_DIR / "pre-testing" / "clamp_range"
sys.path.insert(0, str(EXPERIMENT_DIR.resolve()))
from dataset_bounds import RESULTS, build_comparison, write_report
from IPython.display import Image, display

comparison, manifest, legacy = build_comparison()
write_report(comparison, manifest, legacy)

## All evaluated prompts

`either_outside_percent` is the fraction of tensor positions where the dataset extends below or above the Parti interval. Positive span change means a wider summed empirical range.

In [ ]:
columns = [
    "dataset",
    "embedding",
    "n",
    "global_min",
    "global_max",
    "span_sum",
    "span_change_percent",
    "either_outside_percent",
]
display(
    comparison[(comparison.comparison == "all") & (comparison.region == "all")][
        columns
    ].round(4)
)

## Equal-count comparisons

Each dataset uses its seeded prefix of at most 1,632 unique prompts, compared with the same number of Parti prompts. These are single reproducible samples; they do not estimate sampling uncertainty.

In [ ]:
display(
    comparison[(comparison.comparison == "matched") & (comparison.region == "all")][
        columns
    ].round(4)
)

## Variable coordinates and encoder blocks

Parti has constant token coordinates whose extreme values dominate the global min/max. These supplementary comparisons show changes that global extremes may hide.

In [ ]:
display(
    comparison[(comparison.comparison == "all") & (comparison.region != "all")][
        [
            "dataset",
            "embedding",
            "region",
            "span_change_percent",
            "either_outside_percent",
        ]
    ].round(4)
)
print("Fresh Parti versus packaged bounds:", legacy)

## Difference heatmaps and bounds growth

In [ ]:
for dataset in manifest:
    if dataset != "parti":
        print(dataset)
        display(Image(filename=str(RESULTS / dataset / "bounds_difference.png")))
        display(Image(filename=str(RESULTS / dataset / "pooled_bounds_difference.png")))
        display(Image(filename=str(RESULTS / dataset / "bounds_growth.png")))

## Interpretation

Read results/REPORT.md for the observed dataset differences, runtime and truncation diagnostics. Larger samples can widen extrema even without a distribution change; use the equal-count table alongside the full/budgeted results. Padding and truncation remain part of the production representation. No clamp defaults are changed automatically.